<font color=skyblue>**Deblurring with blur kernels version 2**</font>

In this GUI-based deblurring app, we will implement a deep learning model to perform image deblurring. The app will allow users to 
- select an image file, 
- apply the deblurring model, 
- select a blur kernel, 
- apply the blur kernel to the image, 
- apply the deblurring model to the blurred image,
- display the results in a user-friendly interface. 

The app is designed to be efficient and responsive, providing a seamless experience for users who want to deblur their images.

In [1]:
import os
import cv2
import torch
import numpy as np
import tkinter as tk
import threading
from torchvision import transforms
from tkinter import filedialog, messagebox, ttk
from PIL import Image, ImageTk
from Deblurring_defs import (
    DeblurCNN, DeblurCNN_RES, DeblurSuperResCNN, NAFNet, psnr
)

# Default pre-trained model path -- override via GUI if needed.
device = 'cpu'
pre_trained_model = '../outputs/pre_trained_DeblurCNN_patch_100.pt'

# ───────────────────────────── blur helpers ──────────────────────────────

def apply_gaussian_blur(img_rgb, sigma):
    return cv2.GaussianBlur(img_rgb, (0, 0), sigmaX=float(sigma))

def disk_kernel(radius):
    size = 2 * radius + 1
    kern = np.zeros((size, size), dtype=np.float32)
    cy, cx = radius, radius
    y, x = np.ogrid[:size, :size]
    mask = (x - cx) ** 2 + (y - cy) ** 2 <= radius ** 2
    kern[mask] = 1.0
    kern /= kern.sum()
    return kern

def apply_defocus_blur(img_rgb, radius):
    return cv2.filter2D(img_rgb, -1, disk_kernel(int(radius)))

# ────────────────────────────── model loader ─────────────────────────────

def load_deblur_model(weights_path, device='cpu'):
    if not os.path.exists(weights_path):
        raise FileNotFoundError(f'No pre-trained model found at: {weights_path}')
    checkpoint = torch.load(weights_path, map_location=device)
    if checkpoint is None:
        raise ValueError('Checkpoint is empty.')
    cls = checkpoint.get('model_class', 'DeblurCNN')
    args = checkpoint.get('model_init_args', {})
    model_map = {
        'DeblurCNN': DeblurCNN,
        'DeblurCNN_RES': DeblurCNN_RES,
        'DeblurSuperResCNN': DeblurSuperResCNN,
        'NAFNet': NAFNet,
    }
    ModelClass = model_map.get(cls, DeblurCNN)
    if cls not in model_map:
        print(f"Unknown model class '{cls}', falling back to DeblurCNN.")
    model = ModelClass(**args).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model, checkpoint

def get_recent_model_candidates(default_model, search_dir='../outputs', limit=8):
    candidates = [default_model] if default_model else []
    if os.path.isdir(search_dir):
        files = []
        for name in os.listdir(search_dir):
            if name.lower().endswith(('.pt', '.pth')):
                full = os.path.join(search_dir, name)
                try:
                    mtime = os.path.getmtime(full)
                except OSError:
                    mtime = 0.0
                files.append((mtime, full))
        files.sort(key=lambda x: x[0], reverse=True)
        candidates += [p for _, p in files[:limit * 2]]
    unique, seen = [], set()
    for p in candidates:
        n = os.path.normpath(p)
        if n not in seen:
            seen.add(n)
            unique.append(p)
        if len(unique) >= limit:
            break
    return unique

# ─────────────────────────────── GUI class ───────────────────────────────

class DeblurTypeApp:
    def __init__(self, root, model=None, checkpoint=None,
                 model_path='', recent_models=None, device='cpu'):
        self.root = root
        self.model = model
        self.checkpoint = checkpoint if checkpoint is not None else {}
        self.device = device
        self.model_path = model_path
        self.recent_models = list(recent_models) if recent_models else []

        self.root.title('Image Deblurring — Gaussian & Defocus')
        self.root.geometry('1340x830')

        self.image_path = None
        self.sharp_np = None
        self.deblur_np = None
        self.sharp_tk = self.blur_tk = self.deblur_tk = None

        self.sigma_var = tk.DoubleVar(value=3.0)
        self.radius_var = tk.IntVar(value=5)
        self.blur_type_var = tk.StringVar(value='gaussian')
        self.status_var = tk.StringVar(value='Select model and image, then click Run Deblur.')
        self.recent_model_var = tk.StringVar(value='')

        model_name = self.checkpoint.get('model_class', 'Not loaded')
        model_epoch = self.checkpoint.get('epoch', 0)

        # ── top info bar ──
        top = tk.Frame(self.root)
        top.pack(fill='x', padx=12, pady=(10, 2))

        self.model_info_label = tk.Label(
            top, text=f'Model: {model_name}    Epochs trained: {model_epoch}',
            font=('Segoe UI', 11, 'bold'))
        self.model_info_label.pack(anchor='w')

        self.model_path_var = tk.StringVar(
            value=f'Weights: {self.model_path if self.model_path else "(not selected)"}')
        tk.Label(top, textvariable=self.model_path_var, anchor='w', fg='gray30').pack(anchor='w')

        # ── recent models row ──
        rf = tk.Frame(top)
        rf.pack(fill='x', pady=(4, 0))
        tk.Label(rf, text='Recent models:').pack(side='left', padx=(0, 6))
        self.recent_combo = ttk.Combobox(rf, textvariable=self.recent_model_var,
                                         state='readonly', width=95)
        self.recent_combo.pack(side='left', padx=(0, 8), fill='x', expand=True)
        self.load_recent_btn = tk.Button(rf, text='Load Recent', width=12,
                                         command=self.load_selected_recent_model)
        self.load_recent_btn.pack(side='left')

        # ── model / image pickers ──
        picker = tk.Frame(self.root)
        picker.pack(fill='x', padx=12, pady=(6, 2))
        tk.Button(picker, text='0) Select Model', width=16,
                  command=self.select_model).grid(row=0, column=0, padx=(0, 8), pady=4)
        tk.Button(picker, text='1) Select Image', width=16,
                  command=self.select_image).grid(row=0, column=1, padx=(0, 8), pady=4)
        self.path_label = tk.Label(picker, text='No image selected', anchor='w', width=90)
        self.path_label.grid(row=0, column=2, sticky='w', padx=4)

        # ── blur type + sliders ──
        bf = tk.LabelFrame(self.root, text='Blur Type & Parameters', padx=8, pady=4)
        bf.pack(fill='x', padx=12, pady=(2, 2))

        tk.Radiobutton(bf, text='Gaussian', variable=self.blur_type_var,
                       value='gaussian', command=self._on_blur_type_change
                       ).grid(row=0, column=0, padx=(0, 6), sticky='w')
        tk.Label(bf, text='SigmaX:').grid(row=0, column=1, sticky='e', padx=(0, 4))
        self.sigma_value_label = tk.Label(bf, text='3.0', width=5)
        self.sigma_value_label.grid(row=0, column=2, sticky='w')
        self.sigma_slider = tk.Scale(bf, from_=0.5, to=15.0, resolution=0.5,
                                     orient='horizontal', length=260, variable=self.sigma_var,
                                     command=self._on_sigma_change)
        self.sigma_slider.grid(row=0, column=3, sticky='w', padx=4)

        tk.Radiobutton(bf, text='Defocus (Disk)', variable=self.blur_type_var,
                       value='defocus', command=self._on_blur_type_change
                       ).grid(row=0, column=4, padx=(28, 6), sticky='w')
        tk.Label(bf, text='Radius:').grid(row=0, column=5, sticky='e', padx=(0, 4))
        self.radius_value_label = tk.Label(bf, text='5', width=4)
        self.radius_value_label.grid(row=0, column=6, sticky='w')
        self.radius_slider = tk.Scale(bf, from_=1, to=30, resolution=1,
                                      orient='horizontal', length=200, variable=self.radius_var,
                                      command=self._on_radius_change)
        self.radius_slider.grid(row=0, column=7, sticky='w', padx=4)

        # ── action buttons ──
        act = tk.Frame(self.root)
        act.pack(fill='x', padx=12, pady=(2, 4))
        self.run_btn = tk.Button(act, text='2) Run Deblur', width=16, command=self.run_deblur)
        self.run_btn.grid(row=0, column=0, padx=(0, 8))
        self.save_btn = tk.Button(act, text='3) Save Deblurred', width=16,
                                  command=self.save_deblurred, state='disabled')
        self.save_btn.grid(row=0, column=1, padx=(0, 8))
        tk.Button(act, text='Exit', width=10, command=self.root.destroy).grid(row=0, column=2)

        # ── 3-panel preview ──
        pv = tk.Frame(self.root)
        pv.pack(fill='both', expand=True, padx=12, pady=4)
        lp = tk.Frame(pv, bd=1, relief='solid')
        mp = tk.Frame(pv, bd=1, relief='solid')
        rp = tk.Frame(pv, bd=1, relief='solid')
        for p in (lp, mp, rp):
            p.pack(side='left', fill='both', expand=True, padx=4)

        self.left_title = tk.Label(lp, text='Blurred Image', font=('Segoe UI', 10, 'bold'))
        self.left_title.pack(pady=(8, 4))
        self.mid_title = tk.Label(mp, text='Deblurred Image', font=('Segoe UI', 10, 'bold'))
        self.mid_title.pack(pady=(8, 4))
        self.right_title = tk.Label(rp, text='Sharp (Original)', font=('Segoe UI', 10, 'bold'))
        self.right_title.pack(pady=(8, 4))

        self.left_img_lbl = tk.Label(lp, text='No preview')
        self.left_img_lbl.pack(fill='both', expand=True, padx=8, pady=(0, 8))
        self.mid_img_lbl = tk.Label(mp, text='No preview')
        self.mid_img_lbl.pack(fill='both', expand=True, padx=8, pady=(0, 8))
        self.right_img_lbl = tk.Label(rp, text='No preview')
        self.right_img_lbl.pack(fill='both', expand=True, padx=8, pady=(0, 8))

        tk.Label(self.root, textvariable=self.status_var, anchor='w', fg='navy'
                 ).pack(fill='x', padx=12, pady=(0, 8))

        self._on_blur_type_change()
        self.refresh_recent_dropdown()
        if self.model_path:
            self.add_recent_model(self.model_path, make_current=True)

    # ── blur type toggle ──

    def _on_blur_type_change(self):
        g = 'normal' if self.blur_type_var.get() == 'gaussian' else 'disabled'
        d = 'normal' if self.blur_type_var.get() == 'defocus' else 'disabled'
        self.sigma_slider.config(state=g)
        self.radius_slider.config(state=d)

    def _on_sigma_change(self, _=None):
        self.sigma_value_label.config(text=f'{self.sigma_var.get():.1f}')

    def _on_radius_change(self, _=None):
        self.radius_value_label.config(text=str(int(self.radius_var.get())))

    # ── recent model helpers ──

    def refresh_recent_dropdown(self):
        if self.recent_models:
            self.recent_combo['values'] = self.recent_models
            if self.recent_model_var.get() not in self.recent_models:
                self.recent_model_var.set(self.recent_models[0])
            self.recent_combo.config(state='readonly')
            self.load_recent_btn.config(state='normal')
        else:
            self.recent_combo['values'] = ['(no recent models)']
            self.recent_model_var.set('(no recent models)')
            self.recent_combo.config(state='disabled')
            self.load_recent_btn.config(state='disabled')

    def add_recent_model(self, model_path, make_current=True):
        if not model_path:
            return
        norm = os.path.normpath(model_path)
        dedup, seen = [norm], {norm}
        for p in self.recent_models:
            n = os.path.normpath(p)
            if n not in seen:
                dedup.append(n)
                seen.add(n)
        self.recent_models = dedup[:10]
        self.refresh_recent_dropdown()
        if make_current:
            self.recent_model_var.set(self.recent_models[0])

    def _apply_loaded_model(self, model, checkpoint, model_path):
        self.model, self.checkpoint, self.model_path = model, checkpoint, model_path
        self.model_info_label.config(
            text=f'Model: {checkpoint.get("model_class","?")}    '
                 f'Epochs trained: {checkpoint.get("epoch", 0)}')
        self.model_path_var.set(f'Weights: {model_path}')
        self.add_recent_model(model_path, make_current=True)
        self.status_var.set('Model loaded. Select image and run deblur.')

    def _load_model_from_path(self, model_path):
        model, ckpt = load_deblur_model(model_path, device=self.device)
        self._apply_loaded_model(model, ckpt, model_path)

    def load_selected_recent_model(self):
        sel = self.recent_model_var.get()
        if not sel or sel == '(no recent models)':
            return
        if not os.path.exists(sel):
            messagebox.showerror('Error', f'File not found:\n{sel}')
            return
        try:
            self._load_model_from_path(sel)
        except Exception as e:
            messagebox.showerror('Model Load Error', str(e))

    def select_model(self):
        path = filedialog.askopenfilename(
            title='Select pre-trained model',
            filetypes=[('Model Files', '*.pt *.pth'), ('All Files', '*.*')])
        if not path:
            return
        try:
            self._load_model_from_path(path)
        except Exception as e:
            messagebox.showerror('Model Load Error', str(e))

    # ── image selection ──

    def select_image(self):
        path = filedialog.askopenfilename(
            title='Select an image',
            filetypes=[('Image Files', '*.png *.jpg *.jpeg *.bmp *.tif *.tiff')])
        if not path:
            return
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            messagebox.showerror('Error', f'Cannot read image:\n{path}')
            return
        self.image_path = path
        self.sharp_np = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        self.deblur_np = None
        self.path_label.config(text=path)
        self.right_title.config(text='Sharp (Original)')
        self._show_in_panel(self.right_img_lbl, self.sharp_np, 'sharp')
        self.left_img_lbl.config(image='', text='Run deblur to preview')
        self.mid_img_lbl.config(image='', text='Run deblur to preview')
        self.left_title.config(text='Blurred Image')
        self.mid_title.config(text='Deblurred Image')
        self.save_btn.config(state='disabled')
        self.status_var.set('Image loaded. Choose blur type, set parameter, then Run Deblur.')

    # ── deblur pipeline ──

    def run_deblur(self):
        if self.model is None:
            messagebox.showwarning('Warning', 'Please select a pre-trained model first.')
            return
        if self.sharp_np is None:
            messagebox.showwarning('Warning', 'Please select an image first.')
            return

        btype = self.blur_type_var.get()
        if btype == 'gaussian':
            sigma = float(self.sigma_var.get())
            blur_np = apply_gaussian_blur(self.sharp_np, sigma)
            param_label = f'Gaussian \u03c3={sigma:.1f}'
        else:
            radius = int(self.radius_var.get())
            blur_np = apply_defocus_blur(self.sharp_np, radius)
            param_label = f'Defocus radius={radius}'

        sharp_t = transforms.ToTensor()(self.sharp_np).unsqueeze(0).to(self.device)
        blur_t  = transforms.ToTensor()(blur_np).unsqueeze(0).to(self.device)

        with torch.no_grad():
            deblur_t = self.model(blur_t)

        blur_t   = blur_t.clamp(0, 1)
        deblur_t = deblur_t.clamp(0, 1)

        psnr_blur   = psnr(sharp_t, blur_t)
        psnr_deblur = psnr(sharp_t, deblur_t)

        def to_np(t):
            return np.clip(t.squeeze(0).cpu().permute(1, 2, 0).numpy(), 0, 1)

        blur_show      = to_np(blur_t)
        self.deblur_np = to_np(deblur_t)

        self.left_title.config(
            text=f'Blurred ({param_label}) \u2014 PSNR: {psnr_blur:.2f} dB')
        self.mid_title.config(text=f'Deblurred \u2014 PSNR: {psnr_deblur:.2f} dB')
        self.right_title.config(text='Sharp (Original)')

        self._show_in_panel(self.left_img_lbl,  blur_show,      'blur')
        self._show_in_panel(self.mid_img_lbl,   self.deblur_np, 'deblur')
        self._show_in_panel(self.right_img_lbl, self.sharp_np,  'sharp')

        self.save_btn.config(state='normal')
        self.status_var.set(
            f'Done \u2014 {param_label}. '
            f'Blur PSNR: {psnr_blur:.2f} dB | Deblur PSNR: {psnr_deblur:.2f} dB. '
            'Change params and re-run anytime.')

    # ── save ──

    def save_deblurred(self):
        if self.deblur_np is None:
            messagebox.showwarning('Warning', 'Run deblur first.')
            return
        btype = self.blur_type_var.get()
        tag = (f'gauss{self.sigma_var.get():.0f}' if btype == 'gaussian'
               else f'defocus{self.radius_var.get()}')
        base = (os.path.splitext(os.path.basename(self.image_path))[0]
                if self.image_path else 'image')
        init = f'{base}_deblurred_{tag}.png'
        save_path = filedialog.asksaveasfilename(
            title='Save deblurred image', defaultextension='.png', initialfile=init,
            filetypes=[('PNG', '*.png'), ('JPEG', '*.jpg *.jpeg'),
                       ('Bitmap', '*.bmp'), ('TIFF', '*.tif *.tiff')])
        if not save_path:
            return
        img_u8 = (np.clip(self.deblur_np, 0, 1) * 255).astype(np.uint8)
        ok = cv2.imwrite(save_path, cv2.cvtColor(img_u8, cv2.COLOR_RGB2BGR))
        if ok:
            self.status_var.set(f'Saved: {save_path}')
        else:
            messagebox.showerror('Error', f'Failed to save:\n{save_path}')

    # ── image rendering helper ──

    def _show_in_panel(self, lbl, np_img, slot):
        disp = np_img
        if disp.dtype != np.uint8:
            disp = (np.clip(disp, 0, 1) * 255).astype(np.uint8)
        h, w = disp.shape[:2]
        scale = min(400 / max(1, w), 440 / max(1, h), 1.0)
        nw, nh = max(1, int(w * scale)), max(1, int(h * scale))
        resized = cv2.resize(disp, (nw, nh), interpolation=cv2.INTER_AREA)
        tk_img = ImageTk.PhotoImage(Image.fromarray(resized))
        lbl.config(image=tk_img, text='')
        if slot == 'sharp':    self.sharp_tk  = tk_img
        elif slot == 'blur':   self.blur_tk   = tk_img
        elif slot == 'deblur': self.deblur_tk = tk_img

# ─────────────────────── thread launcher ────────────────────────────────

def _run_gui_thread():
    model, checkpoint, model_path = None, {}, ''
    recent_models = get_recent_model_candidates(
        pre_trained_model, search_dir='../outputs', limit=8)

    if pre_trained_model and os.path.exists(pre_trained_model):
        try:
            model, checkpoint = load_deblur_model(pre_trained_model, device=device)
            model_path = pre_trained_model
        except Exception as e:
            print(f'Could not auto-load default model: {e}')
            print('Use Select Model or Recent Models in the GUI.')
    else:
        print('Default model not found. Use Select Model or Recent Models in the GUI.')

    root = tk.Tk()
    DeblurTypeApp(root, model=model, checkpoint=checkpoint,
                  model_path=model_path, recent_models=recent_models, device=device)
    root.mainloop()


def launch_deblur_gui(blocking=False):
    if blocking:
        _run_gui_thread()
        return None
    t = threading.Thread(target=_run_gui_thread, daemon=True)
    t.start()
    print('DeblurType GUI launched in background — cell finishes immediately.')
    print('Supports Gaussian (sigma) and Defocus/Disk (radius) blur types.')
    print('If no window appears, check if it opened behind other windows.')
    return t


GUI_THREAD = launch_deblur_gui(blocking=False)


DeblurType GUI launched in background — cell finishes immediately.
Supports Gaussian (sigma) and Defocus/Disk (radius) blur types.
If no window appears, check if it opened behind other windows.
